# Fluctuation Diagnostics Analysis

This notebook demonstrates the VAFT Mirnov fluctuation pipeline from a diagnostics ODS: native-resolution raw voltage traces, MATLAB-compatible spectrograms, and toroidal mode-number analysis.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
from omas import ODS, load_omas_json

import vaft.plot as vplot
from vaft.machine_mapping.magnetics import vfit_magnetics_for_shot
from vaft.machine_mapping.utils import get_path, set_path
from vaft.process.magnetics import toroidal_mode_analysis

plt.rcParams["figure.dpi"] = 120
output_dir = Path("outputs/fluctuation_diagnostics_analysis")
output_dir.mkdir(parents=True, exist_ok=True)

## Load Diagnostics ODS

Set `VAFT_DIAGNOSTICS_ODS` to a Snakemake output such as `/srv/vest.filedb/public/39915/omas/39915_diagnostics.json`. The fallback path below is only a convenient local default.

In [ ]:
ods_path = Path(os.environ.get("VAFT_DIAGNOSTICS_ODS", "/srv/vest.filedb/public/39915/omas/39915_diagnostics.json"))
if ods_path.exists():
    ods = load_omas_json(str(ods_path), consistency_check=False)
    source = ods_path
else:
    sample_template = "../vaft/data/legacy/shot_{shot}.json.gz"
    if not Path(sample_template.format(shot=44740)).exists():
        sample_template = "vaft/data/legacy/shot_{shot}.json.gz"
    os.environ["VAFT_RAW_SAMPLE_PATH"] = sample_template
    os.environ["VAFT_RAW_OFFLINE_ONLY"] = "1"
    ods = ODS()
    vfit_magnetics_for_shot(ods, shot=44740, tstart=0.26, tend=0.34, dt=4e-5)
    source = "generated from vaft/data/legacy/shot_44740.json.gz"
source

## Raw Mirnov Voltage Traces

The MATLAB default rows `15` and `38` are one-based rows in the legacy `md` array. In this ODS they correspond to zero-based `b_field_pol_probe` channels `14` and `37`.

In [ ]:
inboard_channel = 14
outboard_channel = 37
time_range = (0.304, 0.330)

fig, ax = vplot.mirnov_signal(
    ods,
    channels=[inboard_channel, outboard_channel],
    time_range=time_range,
    preprocess=False,
    show=False,
)
ax.set_title("Raw Mirnov voltage")
fig.savefig(output_dir / "mirnov_raw_voltage.png", dpi=200, bbox_inches="tight")
fig

## Spectrograms

The default spectrogram settings follow `vest_mirnov.m`: Hann window, `window_size=500`, `time_resolution=1`, and nominal `sample_rate=250e3` inferred from the raw timebase.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
vplot.mirnov_spectrogram(
    ods,
    channel=inboard_channel,
    time_range=time_range,
    max_frequency=80e3,
    ax=axes[0],
    show=False,
)
axes[0].set_title("Inboard midplane Mirnov")
vplot.mirnov_spectrogram(
    ods,
    channel=outboard_channel,
    time_range=time_range,
    max_frequency=80e3,
    ax=axes[1],
    show=False,
)
axes[1].set_title("Outboard midplane Mirnov")
fig.tight_layout()
fig.savefig(output_dir / "mirnov_spectrogram.png", dpi=200, bbox_inches="tight")
fig

## Toroidal Phase Mode Fit

The phase-reference set is stored as voltage-only entries appended to `magnetics.b_field_pol_probe`. When all four reference channels are available, this cell fits wrapped `phase = phase0 - n * phi` lines to the selected time slice. If the local sample lacks the reference signals, the same routine is demonstrated on a synthetic four-channel mode.

In [ ]:
def _has_voltage_data(ods, channel):
    try:
        return np.asarray(get_path(ods, f"magnetics.b_field_pol_probe.{channel}.voltage.data")).size > 0
    except Exception:
        return False

phase_channels = [64, 65, 66, 67]
fit_ods = ods
fit_time = 0.3215
fit_time_range = (fit_time - 0.006, fit_time + 0.006)
fit_frequencies = None

if not all(_has_voltage_data(ods, channel) for channel in phase_channels):
    sample_rate = 250_000.0
    fit_time = 0.3215
    time = np.arange(int(0.04 * sample_rate), dtype=float) / sample_rate + 0.305
    angles = np.deg2rad([0.0, 120.0, 180.0, 240.0])
    fit_ods = {}
    for index, angle in enumerate(angles):
        signal = np.sin(2 * np.pi * 26_000.0 * time + 0.5 - 1 * angle)
        signal += 0.7 * np.sin(2 * np.pi * 52_000.0 * time - 0.2 - 2 * angle)
        set_path(fit_ods, f"magnetics.b_field_pol_probe.{index}.name", f"Synthetic phi={np.rad2deg(angle):.0f}")
        set_path(fit_ods, f"magnetics.b_field_pol_probe.{index}.toroidal_angle", angle)
        set_path(fit_ods, f"magnetics.b_field_pol_probe.{index}.voltage.time", time)
        set_path(fit_ods, f"magnetics.b_field_pol_probe.{index}.voltage.data", signal)
    phase_channels = [0, 1, 2, 3]
    fit_time_range = (fit_time - 0.006, fit_time + 0.006)
    fit_frequencies = [26_000.0, 52_000.0]

fig, ax, phase_fit = vplot.toroidal_phase_mode_fit(
    fit_ods,
    center_time=fit_time,
    channels=phase_channels,
    time_range=fit_time_range,
    frequencies=fit_frequencies,
    num_modes=2,
    candidate_n=range(0, 5),
    window_size=500,
    preprocess=True,
    show=False,
    save_path=output_dir / "toroidal_phase_mode_fit.png",
    return_result=True,
)
fig

In [ ]:
[(mode.frequency / 1e3, mode.n, np.rad2deg(mode.rms_error)) for mode in phase_fit.modes]